# Day 1 AM — Exercise solutions

Instructor / self-check only. Learners should attempt the **Exercises** section in [`_DEMONSTRATIONS.ipynb`](_DEMONSTRATIONS.ipynb) first.

**How to run:** execute `_DEMONSTRATIONS.ipynb` from the top through §06 (so `TOOLS`, `make_model`, `build_budget_graph`, `try_grounded_decision`, and budgets exist), then run this notebook in the **same kernel**.

## Exercise AM-1 — Force a missing FNOL lookup

**Concept:** tool contracts return authoritative records; `found: false` is a valid observation, not an excuse to invent.

**Task:** Change the forced-tool demonstration so it asks for **CLN-007** instead of CLN-001.

**Expected different result:** the tool result shows the email was not found (or has no usable claim/policy IDs). You must **not** build a completed grounded `TriageDecision` from empty evidence.

**TODO hints (from the demo notebook):**
- TODO: find the forced `bind_tools` / `tool_choice` invoke that currently targets CLN-001
- TODO: change only the operator request identifier
- TODO: print the raw tool JSON and assert it does not contain a real claim id
- TODO: optionally call your grounding helper and show `status: refused` (or equivalent)

In [ ]:
from IPython.display import JSON, display
from langchain_core.messages import HumanMessage, SystemMessage

forced_missing = make_model().bind_tools(TOOLS, tool_choice="any").invoke(
    [
        SystemMessage(content="Select exactly one domain tool. Do not answer from memory."),
        HumanMessage(content="Retrieve FNOL email CLN-007."),
    ]
)
call_missing = forced_missing.tool_calls[0]
observed_missing = TOOLS[[t.name for t in TOOLS].index(call_missing["name"])].invoke(
    call_missing["args"]
)
payload_missing = json.loads(observed_missing)

has_claim_id = bool(payload_missing.get("claim_number_ground_truth"))
assert payload_missing.get("found") is False or not has_claim_id, payload_missing

refused = try_grounded_decision("CLN-007")
assert refused["status"] == "refused", refused

display(
    JSON(
        {
            "tool_call": call_missing,
            "tool_result": payload_missing,
            "grounding": refused,
        }
    )
)

## Exercise AM-2 — Relax the model-call budget

**Concept:** application-owned stop conditions (budgets) change observable `stop_reason` and how far the agent loop proceeds.

**Task:** Rebuild the guarded budget graph from §06, but allow **3** model calls instead of **1**, with the same “call all three tools” system prompt for CLN-001.

**Expected different result:** you should **not** stop primarily for `model_call_budget_exhausted` after a single model turn. Compare `model_calls`, `stop_reason`, and whether tool messages appear.

**TODO hints (from the demo notebook):**
- TODO: copy `build_budget_graph` usage from the failure-modes cell
- TODO: change only the budget argument (1 → 3)
- TODO: keep recursion_limit high enough that the budget—not recursion—is the interesting control
- TODO: display a small JSON diff: `{budget_1: ..., budget_3: ...}`

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

budget_prompt = {
    "messages": [
        SystemMessage(
            content=(
                "Always call fnol_lookup, claim_lookup, and policy_lookup before answering. "
                "Keep requesting tools until all three exist."
            )
        ),
        HumanMessage(content="Fully investigate CLN-001."),
    ],
    "model_calls": 0,
    "stop_reason": "not_started",
}


def summarize_budget_run(max_model_calls: int) -> dict:
    result = build_budget_graph(max_model_calls=max_model_calls).invoke(
        dict(budget_prompt),
        config={"recursion_limit": 8},
    )
    tool_messages = [m for m in result["messages"] if isinstance(m, ToolMessage)]
    return {
        "max_model_calls": max_model_calls,
        "model_calls": result["model_calls"],
        "stop_reason": result["stop_reason"],
        "tool_message_count": len(tool_messages),
    }


budget_1 = summarize_budget_run(1)
budget_3 = summarize_budget_run(3)

assert budget_1["stop_reason"] == "model_call_budget_exhausted"
assert budget_3["model_calls"] >= budget_1["model_calls"]
assert budget_3["stop_reason"] != "model_call_budget_exhausted" or budget_3["model_calls"] > 1

display(JSON({"budget_1": budget_1, "budget_3": budget_3}))